# Notebook 4: Casos Avanzados e Integración

**Módulo:** Data Structures - Advanced  
**Objetivo:** Casos complejos e integración con otros módulos  
**Duración estimada:** 30 minutos

---

## Contenido

1. [Setup](#setup)
2. [Múltiples Estructuras](#múltiples-estructuras)
3. [Integración con Analyzer](#integración-con-analyzer)
4. [Integración con Patterns](#integración-con-patterns)
5. [Análisis Completo](#análisis-completo)
6. [Casos Especiales](#casos-especiales)
7. [Performance](#performance)

---

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '../..')

from app.core.parser import parse_pseudocode
from app.core.data_structures import identify_structures, analyze_structure_usage
from app.core.analyzer.analyzer_engine import AnalyzerEngine
from app.core.patterns import detect_patterns

import pandas as pd
import time

print("Setup completado")

---

## 2. Múltiples Estructuras

### 2.1 Algoritmo con Varias Estructuras

In [ ]:
code_multi = """
algorithm topologicalSort(graph, n)
begin
    inDegree ← createArray(n)
    queue ← createQueue()
    result ← createArray(0)
    
    for i ← 1 to n do
        inDegree[i] ← 0
    
    for u ← 1 to n do
        for v in graph[u] do
            inDegree[v] ← inDegree[v] + 1
    
    for i ← 1 to n do
        if inDegree[i] = 0 then
            call enqueue(queue, i)
    
    while not isEmpty(queue) do
    begin
        u ← dequeue(queue)
        call append(result, u)
        
        for v in graph[u] do
        begin
            inDegree[v] ← inDegree[v] - 1
            
            if inDegree[v] = 0 then
                call enqueue(queue, v)
        end
    end
    
    if length(result) != n then
        return "Ciclo detectado"
    
    return result
end
"""

ast = parse_pseudocode(code_multi)
result = identify_structures(ast, min_confidence=0.25)

print("ORDENAMIENTO TOPOLÓGICO")
print(f"Estructuras detectadas: {result.structure_count}\n")

# Crear tabla comparativa
structures_data = []
for struct in result.structures_found:
    structures_data.append({
        "Estructura": struct.structure_name,
        "Confianza": f"{struct.confidence:.2%}",
        "Nivel": struct.confidence_level.value,
        "Variables": ", ".join(struct.variables[:2])
    })

df_structures = pd.DataFrame(structures_data)
print(df_structures.to_string(index=False))

### 2.2 Análisis Individual

In [ ]:
print("\nANÁLISIS INDIVIDUAL DE CADA ESTRUCTURA")

for struct in result.structures_found:
    print(f" {struct.structure_name} (confianza: {struct.confidence:.2%})")
    
    # Indicadores
    print("\n Indicadores encontrados:")
    for ind in struct.indicators_found[:3]:
        print(f"   • {ind.name}: {ind.evidence}")
    
    # Operaciones
    if struct.operations:
        print(f"\n Operaciones ({len(struct.operations)}):")
        for op in struct.operations[:3]:
            print(f"   • {op}")
    
    # Uso
    usage = analyze_structure_usage(ast, struct)
    print(f"\n Uso:")
    print(f"   • Total operaciones: {usage.total_operations}")
    print(f"   • Patrón: {usage.access_pattern}")

### 2.3 Estructura Principal vs Secundarias

In [ ]:
print("\nESTRUCTURA PRINCIPAL")

if result.primary_structure:
    primary = result.primary_structure
    print(f"Nombre: {primary.structure_name}")
    print(f"Confianza: {primary.confidence:.2%}")
    print(f"\n¿Por qué es la principal?")
    print(f"  • Mayor confianza entre todas")
    print(f"  • Indicadores: {len(primary.indicators_found)}/{len(primary.indicators_found) + len(primary.indicators_missing)}")

print("\nESTRUCTURAS SECUNDARIAS")

for i, struct in enumerate(result.structures_found[1:], 1):
    print(f"{i}. {struct.structure_name}: {struct.confidence:.2%}")

---

## 3. Integración con Analyzer

### 3.1 Análisis Completo: Complejidad + Estructuras

In [ ]:
code_example = """
algorithm mergeSort(A[n])
begin
    if n > 1 then
    begin
        mid ← n / 2
        left ← A[1..mid]
        right ← A[mid+1..n]
        
        call mergeSort(left)
        call mergeSort(right)
        call merge(A, left, right)
    end
end
"""

print(" ANÁLISIS COMPLETO: MERGE SORT")

ast = parse_pseudocode(code_example)

# 1. Análisis de Complejidad
engine = AnalyzerEngine()
complexity_result = engine.analyze(ast, analyze_line_by_line=False)

print("\n COMPLEJIDAD TEMPORAL:")
print(f"   • Big O: {complexity_result.big_o}")
print(f"   • Omega: {complexity_result.omega}")
print(f"   • Theta: {complexity_result.theta}")

# 2. Estructuras de Datos
structures_result = identify_structures(ast)

print("\nESTRUCTURAS DETECTADAS:")
if structures_result.primary_structure:
    print(f"   • Principal: {structures_result.primary_structure.structure_name}")
    print(f"   • Confianza: {structures_result.primary_structure.confidence:.2%}")

# 3. Conclusión Integrada
print("\n CONCLUSIÓN INTEGRADA:")
print(f"   • Algoritmo: {complexity_result.algorithm_name}")
print(f"   • Complejidad: {complexity_result.big_o}")
print(f"   • Estructura: {structures_result.primary_structure.structure_name}")
print(f"   • Tipo: Recursivo" if complexity_result.is_recursive else "   • Tipo: Iterativo")

### 3.2 Relación Estructura-Complejidad

In [ ]:
# Analizar cómo la estructura afecta la complejidad
print("\n RELACIÓN ESTRUCTURA-COMPLEJIDAD")

if structures_result.primary_structure:
    struct = structures_result.primary_structure
    usage = analyze_structure_usage(ast, struct)
    
    print(f"\nEstructura: {struct.structure_name}")
    print(f"Patrón de acceso: {usage.access_pattern}")
    
    if usage.complexity_impact:
        print("\nImpacto en complejidad:")
        for key, value in usage.complexity_impact.items():
            print(f"  • {key}: {value}")
    
    print(f"\nComplejidad temporal del algoritmo: {complexity_result.big_o}")
    print(f"Complejidad espacial: {complexity_result.space_analysis.space_complexity if complexity_result.space_analysis else 'N/A'}")

---

## 4. Integración con Patterns

### 4.1 Estructuras + Patrones

In [ ]:
code_dp = """
algorithm fibonacci(n)
begin
    dp ← createArray(n+1)
    dp[0] ← 0
    dp[1] ← 1
    
    for i ← 2 to n do
        dp[i] ← dp[i-1] + dp[i-2]
    
    return dp[n]
end
"""

print("ANÁLISIS: ESTRUCTURAS + PATRONES")

ast = parse_pseudocode(code_dp)

# 1. Detectar Patrones
patterns_result = detect_patterns(ast, min_confidence=0.3)

print("\n PATRÓN ALGORÍTMICO:")
if patterns_result.primary_pattern:
    print(f"   • Patrón: {patterns_result.primary_pattern.pattern.pattern_name}")
    print(f"   • Confianza: {patterns_result.primary_pattern.pattern.confidence:.2%}")

# 2. Detectar Estructuras
structures_result = identify_structures(ast)

print("\n ESTRUCTURA:")
if structures_result.primary_structure:
    print(f"   • Estructura: {structures_result.primary_structure.structure_name}")
    print(f"   • Confianza: {structures_result.primary_structure.confidence:.2%}")

# 3. Relación
print("\n RELACIÓN PATRÓN-ESTRUCTURA:")
print("   • Programación Dinámica típicamente usa arrays/tablas")
print("   • El array 'dp' almacena subproblemas resueltos")
print("   • Evita recálculos (memoización)")

### 4.2 Tabla de Relaciones

In [ ]:
# Relaciones típicas patrón-estructura
relations = {
    "Fuerza Bruta": ["Array"],
    "Recursión": ["Stack (implícita)"],
    "Divide y Vencerás": ["Array", "Tree"],
    "Programación Dinámica": ["Array", "Matrix", "Hash Table"],
    "Greedy": ["Array", "Priority Queue", "Heap"],
    "Backtracking": ["Stack", "Tree"],
    "Branch and Bound": ["Priority Queue", "Stack"],
    "Sorting": ["Array"],
    "Searching": ["Array", "Tree", "Graph"]
}

print("\nRELACIONES PATRÓN-ESTRUCTURA")

for pattern, structures in relations.items():
    print(f"\n{pattern}:")
    for struct in structures:
        print(f"  • {struct}")

---

## 5. Análisis Completo

### 5.1 Función de Análisis Total

In [ ]:
def analyze_algorithm_complete(code: str):
    """
    Análisis completo de un algoritmo:
    - Parsing
    - Complejidad temporal y espacial
    - Patrones algorítmicos
    - Estructuras de datos
    """
    print("ANÁLISIS COMPLETO DE ALGORITMO")
    
    # 1. Parse
    ast = parse_pseudocode(code)
    print(f"Código parseado: {ast.algorithm.name}\n")
    
    # 2. Complejidad
    engine = AnalyzerEngine()
    complexity = engine.analyze(ast, analyze_line_by_line=False)
    
    print("COMPLEJIDAD:")
    print(f"   • Temporal (O): {complexity.big_o}")
    print(f"   • Temporal (Ω): {complexity.omega}")
    print(f"   • Temporal (Θ): {complexity.theta}")
    if complexity.space_analysis:
        print(f"   • Espacial: {complexity.space_analysis.space_complexity}")
    print(f"   • Es recursivo: {'Sí' if complexity.is_recursive else 'No'}")
    
    # 3. Patrones
    patterns = detect_patterns(ast, min_confidence=0.3)
    
    print(f"\nPATRONES DETECTADOS: {patterns.pattern_count}")
    if patterns.primary_pattern:
        primary_pattern = patterns.primary_pattern.pattern
        print(f"   • Principal: {primary_pattern.pattern_name}")
        print(f"   • Confianza: {primary_pattern.confidence:.2%}")
    
    # 4. Estructuras
    structures = identify_structures(ast, min_confidence=0.3)
    
    print(f"\nESTRUCTURAS DETECTADAS: {structures.structure_count}")
    if structures.primary_structure:
        print(f"   • Principal: {structures.primary_structure.structure_name}")
        print(f"   • Confianza: {structures.primary_structure.confidence:.2%}")
    
    # 5. Resumen
    print("\nRESUMEN:")
    print(f"   Algoritmo: {ast.algorithm.name}")
    print(f"   Complejidad: {complexity.big_o}")
    print(f"   Patrón: {primary_pattern.pattern_name if patterns.primary_pattern else 'N/A'}")
    print(f"   Estructura: {structures.primary_structure.structure_name if structures.primary_structure else 'N/A'}")
    
    return {
        "complexity": complexity,
        "patterns": patterns,
        "structures": structures
    }

### 5.2 Ejemplo de Uso

In [ ]:
code_quicksort = """
algorithm quicksort(A[n], low, high)
begin
    if low < high then
    begin
        pivot ← partition(A, low, high)
        call quicksort(A, low, pivot-1)
        call quicksort(A, pivot+1, high)
    end
end
"""

# Análisis completo
results = analyze_algorithm_complete(code_quicksort)

---

## 6. Casos Especiales

### 6.1 Estructuras Anidadas

In [ ]:
code_nested = """
algorithm matrixMultiply(A[n][n], B[n][n])
begin
    C ← createMatrix(n, n)
    
    for i ← 1 to n do
        for j ← 1 to n do
        begin
            C[i][j] ← 0
            for k ← 1 to n do
                C[i][j] ← C[i][j] + A[i][k] * B[k][j]
        end
    
    return C
end
"""

print("CASO: MATRIZ (ARRAY 2D)")

ast = parse_pseudocode(code_nested)
result = identify_structures(ast)

if result.primary_structure:
    struct = result.primary_structure
    print(f"Estructura detectada: {struct.structure_name}")
    print(f"Propiedades:")
    for k, v in struct.properties.items():
        print(f"  • {k}: {v}")

### 6.2 Estructuras Implícitas

In [ ]:
# Recursión = Stack implícita
code_recursive = """
algorithm towerOfHanoi(n, from, to, aux)
begin
    if n > 0 then
    begin
        call towerOfHanoi(n-1, from, aux, to)
        move disk from 'from' to 'to'
        call towerOfHanoi(n-1, aux, to, from)
    end
end
"""

print("\nCASO: ESTRUCTURA IMPLÍCITA")

ast = parse_pseudocode(code_recursive)
result = identify_structures(ast)

if result.primary_structure:
    struct = result.primary_structure
    print(f"Estructura: {struct.structure_name}")
    
    if struct.properties.get('is_implicit_stack'):
        print("Pila implícita detectada (recursión)")

### 6.3 Sin Estructura Clara

In [ ]:
code_math = """
algorithm gcd(a, b)
begin
    while b != 0 do
    begin
        temp ← b
        b ← a mod b
        a ← temp
    end
    return a
end
"""

print("\n CASO: SIN ESTRUCTURA DE DATOS COMPLEJA")

ast = parse_pseudocode(code_math)
result = identify_structures(ast, min_confidence=0.2)

if result.structure_count == 0:
    print("No se detectaron estructuras de datos complejas")
    print("Algoritmo usa solo variables simples")
else:
    print(f"Estructuras detectadas: {result.structure_count}")

---

## 7. Performance

### 7.1 Benchmark de Detección

In [ ]:
def benchmark_detection(code: str, iterations: int = 10):
    """Mide tiempo de detección"""
    ast = parse_pseudocode(code)
    
    times = []
    for _ in range(iterations):
        start = time.time()
        result = identify_structures(ast)
        elapsed = time.time() - start
        times.append(elapsed)
    
    avg_time = sum(times) / len(times)
    min_time = min(times)
    max_time = max(times)
    
    return {
        "avg": avg_time,
        "min": min_time,
        "max": max_time,
        "iterations": iterations
    }

# Test con diferentes tamaños
print("BENCHMARK DE DETECCIÓN")

# Código pequeño
code_small = """
algorithm test(A[10])
begin
    for i ← 1 to 10 do
        x ← A[i]
end
"""

perf_small = benchmark_detection(code_small)
print(f"\nCódigo pequeño (10 elementos):")
print(f"  • Promedio: {perf_small['avg']*1000:.2f} ms")
print(f"  • Mínimo: {perf_small['min']*1000:.2f} ms")
print(f"  • Máximo: {perf_small['max']*1000:.2f} ms")

---

## 8. Ejercicio Final

Analiza completamente el algoritmo de Dijkstra:

In [ ]:
code_dijkstra = """
algorithm dijkstra(graph, source, n)
begin
    dist ← createArray(n)
    visited ← createArray(n)
    pq ← createMinHeap()
    
    for i ← 1 to n do
    begin
        dist[i] ← infinity
        visited[i] ← false
    end
    
    dist[source] ← 0
    call insert(pq, source, 0)
    
    while not isEmpty(pq) do
    begin
        u ← extractMin(pq)
        
        if visited[u] then
            continue
        
        visited[u] ← true
        
        for v in graph[u] do
        begin
            alt ← dist[u] + weight(u, v)
            
            if alt < dist[v] then
            begin
                dist[v] ← alt
                call insert(pq, v, alt)
            end
        end
    end
    
    return dist
end
"""

print("EJERCICIO FINAL: DIJKSTRA")

# Tu análisis aquí
results_dijkstra = analyze_algorithm_complete(code_dijkstra)

print("\nPREGUNTAS:")
print("1. ¿Cuántas estructuras diferentes se usan?")
print("2. ¿Cuál es la complejidad temporal?")
print("3. ¿Qué patrón algorítmico se detectó?")
print("4. ¿Por qué Dijkstra usa un heap?")

---

## Resumen

Has aprendido:
- Analizar algoritmos con múltiples estructuras
- Integrar con módulos de complejidad y patrones
- Realizar análisis completos
- Manejar casos especiales
- Evaluar performance del detector

**¡Módulo completado!** 

Ahora puedes:
- Detectar estructuras en cualquier algoritmo
- Analizar su uso y patrones
- Integrar con análisis de complejidad
- Generar recomendaciones de optimización

---

## Recursos Adicionales

- [Documentación Completa](../../docs/DATA_STRUCTURES.md)
- [API Reference](../../docs/api_reference.md)
- [Tests Unitarios](../../tests/unit/test_data_structures.py)

**Próximos pasos:** Explorar la API REST y crear tu propia aplicación!